In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, Dropout, BatchNormalization

2025-04-24 00:14:02.461279: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-24 00:14:02.496480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745442842.535409   35793 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745442842.546808   35793 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-24 00:14:02.586773: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
FOLDER_PATH = "../../img/ote"
FRAME_COUNT = 16
IMG_SIZE = 64
CLASS_COUNT = 3 

In [3]:
class_names = ["0","1","2"]

In [ ]:
def load_videos(folder, frame_count=FRAME_COUNT, img_size=IMG_SIZE):
    videos = []
    labels = []

    for video_folder in os.listdir(folder):
        video_path = os.path.join(folder, video_folder)
        if os.path.isdir(video_path):
            try:
                video_id, label = video_folder.split("_")
                label = int(label[1])
            except ValueError:
                print(f"[SKIP] invalid dir name: {video_folder}")
                continue

            frames = []
            frame_files = sorted(os.listdir(video_path))[:frame_count]
            print(f"[INFO] {video_folder} -> {len(frame_files)} frame")

            for filename in frame_files:
                img_path = os.path.join(video_path, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    frames.append(img)
                else:
                    print(f"[WARN] frame can not be read: {img_path}")

            if len(frames) == frame_count:
                video_array = np.array(frames).reshape(frame_count, img_size, img_size, 1)
                videos.append(video_array)
                labels.append(label)
            else:
                print(f"[SKIP] {video_folder} -> not enough frame ({len(frames)}/{frame_count})")

    print(f"[SUMMARY] sum of videos: {len(videos)}")
    return np.array(videos), np.array(labels)


X, y = load_videos(FOLDER_PATH, FRAME_COUNT, IMG_SIZE)
y = to_categorical(y, num_classes=CLASS_COUNT)

print("Number of videos:", len(X)) 
print("Shape of data:", X.shape) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


[INFO] 105_100 -> 16 frame
[INFO] 114_000 -> 16 frame
[INFO] 117_100 -> 16 frame
[INFO] 140_000 -> 16 frame
[INFO] 130_000 -> 16 frame
[INFO] 136_000 -> 16 frame
[INFO] 110_000 -> 16 frame
[INFO] 106_111 -> 16 frame
[INFO] 107_000 -> 16 frame
[INFO] 128_200 -> 0 frame
[SKIP] 128_200 -> Yeterli frame yok (0/16)
[INFO] 134_000 -> 16 frame
[INFO] 113_000 -> 16 frame
[INFO] 123_000 -> 16 frame
[INFO] 125_100 -> 0 frame
[SKIP] 125_100 -> Yeterli frame yok (0/16)
[INFO] 132_000 -> 16 frame
[INFO] 131_101 -> 16 frame
[INFO] 119_010 -> 16 frame
[INFO] 118_000 -> 16 frame
[INFO] 138_100 -> 16 frame
[INFO] 120_000 -> 16 frame
[INFO] 104_000 -> 16 frame
[INFO] 139_110 -> 16 frame
[INFO] 141_000 -> 16 frame
[INFO] 116_000 -> 16 frame
[INFO] 137_000 -> 16 frame
[INFO] 124_000 -> 16 frame
[INFO] 101_000 -> 0 frame
[SKIP] 101_000 -> Yeterli frame yok (0/16)
[INFO] 129_100 -> 16 frame
[INFO] 126_100 -> 0 frame
[SKIP] 126_100 -> Yeterli frame yok (0/16)
[INFO] 122_000 -> 16 frame
[INFO] 127_000 -> 0 fr

In [5]:
print(f"Number of train sets {len(X_train)}")
print(f"Number of test sets {len(X_test)}")

Number of train sets 28
Number of test sets 8


In [6]:
input_shape = (FRAME_COUNT, IMG_SIZE, IMG_SIZE, 1)

model = Sequential([
    Conv3D(32, kernel_size=(3,3,3), activation='relu',padding="same", input_shape=input_shape),
    MaxPooling3D(pool_size=(2,2,2)),
    BatchNormalization(),

    Conv3D(64, kernel_size=(3,3,3),padding="same", activation='relu'),
    MaxPooling3D(pool_size=(2,2,2)),
    BatchNormalization(),

    Conv3D(64, kernel_size=(3,3,3), padding="same",activation='relu'),
    MaxPooling3D(pool_size=(2,2,2)),
    BatchNormalization(),

    Flatten(),
    Dense(32, activation='relu'),

    Dense(128, activation='relu'),

    Dense(64, activation='relu'),
    Dropout(0.1),
    Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(X_train, y_train, epochs=10, batch_size=4, validation_split=0.1)

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.2f}")

/home/tolgahan/Desktop/machine-learning/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-04-24 00:14:07.424145: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 16, 64, 64, 32) │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 8, 32, 32, 32)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 32, 32, 32)  │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 8, 32, 32, 64)  │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 4, 16, 16, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 4, 16, 16, 64)  │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 4, 16, 16, 64)  │       110,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_2 (MaxPooling3D)  │ (None, 2, 8, 8, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 2, 8, 8, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       262,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 442,403 (1.69 MB)

 Trainable params: 442,083 (1.69 MB)

 Non-trainable params: 320 (1.25 KB)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 369ms/step - accuracy: 0.5208 - loss: 1.0414 - val_accuracy: 1.0000 - val_loss: 0.0023
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 275ms/step - accuracy: 0.8537 - loss: 0.6413 - val_accuracy: 1.0000 - val_loss: 7.1526e-07
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 258ms/step - accuracy: 0.8360 - loss: 0.3960 - val_accuracy: 1.0000 - val_loss: 2.7815e-07
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 259ms/step - accuracy: 0.8397 - loss: 0.5072 - val_accuracy: 1.0000 - val_loss: 3.9736e-08
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 256ms/step - accuracy: 0.8032 - loss: 0.3917 - val_accuracy: 1.0000 - val_loss: 2.7815e-07
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 256ms/step - accuracy: 0.9633 - loss: 0.2259 - val_accuracy: 1.0000 - val_loss: 2.6186e-05
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 264ms/step - accuracy: 0.8991 - loss: 0.2074 - val_accuracy: 1.0000 - val_loss: 0.0075
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 262ms/step - accuracy: 0.9154 - loss: 0.2253 - val_accuracy